# Data Cleaning and Initial Inspection 
## Miami International Airport (MIA) | January–February 2024

This notebook covers all data preparation steps:
- Initial inspection of the raw dataset
- Duplicate removal
- Missing value analysis and handling
- Data type conversion
- Filtering for MIA origin flights
- Export of cleaned dataset

NOTE: The dataset covers only January–February 2024.
All findings reflect winter-season operational patterns only.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

from my_modules.data_collection_module import load_flight_data

print("Libraries loaded.")


Libraries loaded.


In [2]:
# Load raw dataset
df = load_flight_data("../data/raw/flight_data_2024.csv")

# Preserve a copy of the raw dataset before any transformation
df_raw = df.copy()

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


Raw dataset shape: (1048575, 18)
Columns: ['year', 'month', 'day_of_month', 'day_of_week', 'fl_date', 'origin', 'origin_city_name', 'origin_state_nm', 'dep_time', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'cancelled', 'air_time', 'distance', 'weather_delay', 'late_aircraft_delay']


,year,month,day_of_month,day_of_week,fl_date,origin,origin_city_name,origin_state_nm,dep_time,taxi_out,wheels_off,wheels_on,taxi_in,cancelled,air_time,distance,weather_delay,late_aircraft_delay
0,2024,1,1,1,1/1/2024,JFK,"New York, NY",New York,1247.0,31.0,1318.0,1442.0,7.0,0,84.0,509,0,0
1,2024,1,1,1,1/1/2024,MSP,"Minneapolis, MN",Minnesota,1001.0,20.0,1021.0,1249.0,6.0,0,88.0,622,0,0
2,2024,1,1,1,1/1/2024,JFK,"New York, NY",New York,1411.0,21.0,1432.0,1533.0,8.0,0,61.0,288,0,0
3,2024,1,1,1,1/1/2024,RIC,"Richmond, VA",Virginia,1643.0,13.0,1656.0,1747.0,12.0,0,51.0,288,0,0
4,2024,1,1,1,1/1/2024,DTW,"Detroit, MI",Michigan,1010.0,21.0,1031.0,1016.0,4.0,0,45.0,237,0,0


## Dataset Overview
The raw dataset contains flight-level operational data for all U.S. airports.
We will filter for Miami International Airport (MIA) after cleaning.

Dataset time coverage: January 1, 2024 – February 29, 2024.
This covers only the first two months of 2024 (winter season).

In [3]:
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")


Total rows: 1,048,575
Total columns: 18


(1048575, 18)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 18 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   year                 1048575 non-null  int64  
 1   month                1048575 non-null  int64  
 2   day_of_month         1048575 non-null  int64  
 3   day_of_week          1048575 non-null  int64  
 4   fl_date              1048575 non-null  str    
 5   origin               1048575 non-null  str    
 6   origin_city_name     1048575 non-null  str    
 7   origin_state_nm      1048575 non-null  str    
 8   dep_time             1026022 non-null  float64
 9   taxi_out             1025450 non-null  float64
 10  wheels_off           1025450 non-null  float64
 11  wheels_on            1024898 non-null  float64
 12  taxi_in              1024898 non-null  float64
 13  cancelled            1048575 non-null  int64  
 14  air_time             1022824 non-null  float64
 15  distance 

In [5]:
df.describe()

,year,month,day_of_month,day_of_week,dep_time,taxi_out,wheels_off,wheels_on,taxi_in,cancelled,air_time,distance,weather_delay,late_aircraft_delay
count,1048575.0,1.048575e+06,1.048575e+06,1.048575e+06,1.026022e+06,1.025450e+06,1.025450e+06,1.024898e+06,1.024898e+06,1.048575e+06,1.022824e+06,1.048575e+06,1.048575e+06,1.048575e+06
mean,2024.0,1.478081e+00,1.530512e+01,3.893483e+00,1.325074e+03,1.825012e+01,1.349996e+03,1.476156e+03,8.082517e+00,2.222635e-02,1.162270e+02,8.345389e+02,1.194321e+00,5.326660e+00
std,0.0,4.995196e-01,8.585503e+00,2.010038e+00,4.972990e+02,1.044025e+01,4.980426e+02,5.198682e+02,6.512591e+00,1.474190e-01,7.091204e+01,5.923104e+02,2.005819e+01,2.975676e+01
min,2024.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,8.000000e+00,3.100000e+01,0.000000e+00,0.000000e+00
25%,2024.0,1.000000e+00,8.000000e+00,2.000000e+00,9.110000e+02,1.200000e+01,9.290000e+02,1.058000e+03,4.000000e+00,0.000000e+00,6.400000e+01,4.020000e+02,0.000000e+00,0.000000e+00
50%,2024.0,1.000000e+00,1.500000e+01,4.000000e+00,1.323000e+03,1.500000e+01,1.337000e+03,1.510000e+03,6.000000e+00,0.000000e+00,1.000000e+02,6.920000e+02,0.000000e+00,0.000000e+00
75%,2024.0,2.000000e+00,2.300000e+01,6.000000e+00,1.736000e+03,2.100000e+01,1.750000e+03,1.914000e+03,9.000000e+00,0.000000e+00,1.470000e+02,1.069000e+03,0.000000e+00,0.000000e+00
max,2024.0,2.000000e+00,3.100000e+01,7.000000e+00,2.400000e+03,2.130000e+02,2.400000e+03,2.400000e+03,4.440000e+02,1.000000e+00,7.230000e+02,5.095000e+03,1.804000e+03,2.100000e+03


## Missing Values
We examine missing values across all columns before making any decisions.
Operational time columns (dep_time, taxi_out, wheels_off, etc.) are expected
to be missing for cancelled flights — these flights never departed and therefore
never generated operational time records.

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

print("Missing values in raw dataset:")
print(missing_summary.to_string())

Missing values in raw dataset:
            Missing Count  Missing %
dep_time            22553       2.15
taxi_out            23125       2.21
wheels_off          23125       2.21
wheels_on           23677       2.26
taxi_in             23677       2.26
air_time            25751       2.46


## Duplicate Records
We check for and remove exact duplicate rows before any other transformation.
Working on a copy ensures the raw dataset remains intact for traceability.

In [7]:
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates:,}")

Duplicate rows found: 7,401


In [8]:
# Work on a clean copy — preserve df_raw untouched
df_clean = df.drop_duplicates().copy()

print(f"Rows before: {df.shape[0]:,}")
print(f"Rows after removing duplicates: {df_clean.shape[0]:,}")
print(f"Duplicates removed: {df.shape[0] - df_clean.shape[0]:,}")

Rows before: 1,048,575
Rows after removing duplicates: 1,041,174
Duplicates removed: 7,401


## Data Type Conversion
We convert fl_date from string to datetime format to enable
time-based operations and proper date filtering.

In [9]:
df_clean['fl_date'] = pd.to_datetime(df_clean['fl_date'])

print(f"fl_date dtype: {df_clean['fl_date'].dtype}")
print(f"Date range: {df_clean['fl_date'].min().date()} to {df_clean['fl_date'].max().date()}")
print("NOTE: Dataset covers only January–February 2024 (winter season).")

fl_date dtype: datetime64[us]
Date range: 2024-01-01 to 2024-02-29
NOTE: Dataset covers only January–February 2024 (winter season).


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 18 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   year                 1048575 non-null  int64  
 1   month                1048575 non-null  int64  
 2   day_of_month         1048575 non-null  int64  
 3   day_of_week          1048575 non-null  int64  
 4   fl_date              1048575 non-null  str    
 5   origin               1048575 non-null  str    
 6   origin_city_name     1048575 non-null  str    
 7   origin_state_nm      1048575 non-null  str    
 8   dep_time             1026022 non-null  float64
 9   taxi_out             1025450 non-null  float64
 10  wheels_off           1025450 non-null  float64
 11  wheels_on            1024898 non-null  float64
 12  taxi_in              1024898 non-null  float64
 13  cancelled            1048575 non-null  int64  
 14  air_time             1022824 non-null  float64
 15  distance 

In [11]:
mia_df = df[df['origin'] == 'MIA']
mia_df.shape

(19470, 18)

## Investigating Missing Values
Missing values in operational columns are not random —
they are structurally caused by cancellations.
Cancelled flights never depart, so dep_time, taxi_out, etc. are always null.
We verify this before deciding how to handle them.

In [12]:
print("Missing values — CANCELLED flights only:")
print(df_clean[df_clean['cancelled'] == 1].isnull().sum()[
    df_clean[df_clean['cancelled'] == 1].isnull().sum() > 0
])

print("\nMissing values — NON-CANCELLED flights only:")
print(df_clean[df_clean['cancelled'] == 0].isnull().sum()[
    df_clean[df_clean['cancelled'] == 0].isnull().sum() > 0
])

Missing values — CANCELLED flights only:
dep_time      15152
taxi_out      15724
wheels_off    15724
wheels_on     15905
taxi_in       15905
air_time      15905
dtype: int64

Missing values — NON-CANCELLED flights only:
wheels_on     371
taxi_in       371
air_time     2445
dtype: int64


Missing values in flight time-related variables are concentrated in cancelled flights, which is expected because cancelled flights do not depart and therefore do not generate operational time metrics.

## Handling Missing Values

Strategy:
- Delay columns (weather_delay, late_aircraft_delay): fill with 0.
  A missing delay value means no delay was recorded — equivalent to zero.
- Operational time columns (dep_time, taxi_out, etc.): missing values
  in cancelled flights are expected and kept as-is.
- Non-cancelled flights with missing dep_time (incomplete records):
  dropped as they cannot be reliably used for modeling.

In [13]:
# Fill delay columns with 0 — missing = no delay recorded
df_clean['weather_delay'] = df_clean['weather_delay'].fillna(0)
df_clean['late_aircraft_delay'] = df_clean['late_aircraft_delay'].fillna(0)

# Drop non-cancelled flights with missing dep_time (incomplete records)
before = df_clean.shape[0]
df_clean = df_clean[
    ~((df_clean['cancelled'] == 0) & (df_clean['dep_time'].isna()))
].copy()
after = df_clean.shape[0]

print(f"Incomplete non-cancelled records dropped: {before - after:,}")
print(f"Remaining rows: {after:,}")

print("\nMissing values after handling:")
remaining = df_clean.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print("No missing values in key modeling columns.")
else:
    print(remaining.to_string())

Incomplete non-cancelled records dropped: 0
Remaining rows: 1,041,174

Missing values after handling:
dep_time      15152
taxi_out      15724
wheels_off    15724
wheels_on     16276
taxi_in       16276
air_time      18350


## Filter for Miami Flights
We filter the national dataset to focus exclusively on flights
departing from Miami International Airport (MIA).

In [14]:
mia_df = df_clean[df_clean['origin'] == 'MIA'].copy()

print(f"MIA flights: {mia_df.shape[0]:,}")
print(f"Date range: {mia_df['fl_date'].min().date()} to {mia_df['fl_date'].max().date()}")
print(f"Cancellation rate: {mia_df['cancelled'].mean():.2%}")
print(f"Delay rate (>15 min): {(mia_df['late_aircraft_delay'] > 15).mean():.2%}")

MIA flights: 19,396
Date range: 2024-01-01 to 2024-02-29
Cancellation rate: 1.07%
Delay rate (>15 min): 11.42%


In [15]:
print("Final dataset summary:")
print(f"  Shape: {mia_df.shape}")
print(f"  Date coverage: January–February 2024 only")
print(f"  Missing values remaining:")

final_missing = mia_df.isnull().sum()
final_missing = final_missing[final_missing > 0]

if len(final_missing) == 0:
    print("  None in key columns.")
else:
    print(final_missing.to_string())

mia_df.info()

Final dataset summary:
  Shape: (19396, 18)
  Date coverage: January–February 2024 only
  Missing values remaining:
dep_time      180
taxi_out      205
wheels_off    205
wheels_on     210
taxi_in       210
air_time      267
<class 'pandas.DataFrame'>
Index: 19396 entries, 21 to 1048569
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   year                 19396 non-null  int64         
 1   month                19396 non-null  int64         
 2   day_of_month         19396 non-null  int64         
 3   day_of_week          19396 non-null  int64         
 4   fl_date              19396 non-null  datetime64[us]
 5   origin               19396 non-null  str           
 6   origin_city_name     19396 non-null  str           
 7   origin_state_nm      19396 non-null  str           
 8   dep_time             19216 non-null  float64       
 9   taxi_out             19191 non-null  float64    

## Save Cleaned Dataset

In [16]:
output_path = "../data/processed/mia_flights_clean.csv"
mia_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {mia_df.shape}")
print("Data cleaning complete.")

Saved: ../data/processed/mia_flights_clean.csv
Shape: (19396, 18)
Data cleaning complete.
